In [1]:
import pandas as pd                                     # pandas: tables
from scipy import stats                                 # scipy.stats: chi-square test
from statsmodels.stats.power import NormalIndPower      # sample-size calculator for yes/no outcomes
from statsmodels.stats.proportion import proportion_effectsize   # turns two percents into an effect size

url = "https://raw.githubusercontent.com/Hindhuja-Gutha/Analyze-AB-TestResults/master/ab_data.csv"   # where the file is on the internet
df = pd.read_csv(url)        # read the file into a table (DataFrame) called df
print(df.shape)              # number of rows and columns
print(df.head())             # the first 5 rows

(294478, 5)
   user_id                   timestamp      group landing_page  converted
0   851104  2017-01-21 22:11:48.556739    control     old_page          0
1   804228  2017-01-12 08:01:45.159739    control     old_page          0
2   661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3   853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4   864975  2017-01-21 01:52:26.210827    control     old_page          1


In [5]:
df.isna().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [6]:
pd.crosstab(df["group"],df["landing_page"])

landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [7]:
df["user_id"].duplicated().sum()

np.int64(3894)

In [11]:
df_clean=df.loc[((df["group"]=="control") & (df["landing_page"]=="old_page") | (df["group"]=="treatment") & (df["landing_page"]=="new_page")), :]

In [12]:
pd.crosstab(df_clean["group"],df_clean["landing_page"])

landing_page,new_page,old_page
group,,
control,0,145274
treatment,145311,0


In [13]:
df_clean["user_id"].duplicated().sum()

np.int64(1)

In [14]:
clean=df_clean.drop_duplicates(subset="user_id")
clean["user_id"].duplicated().sum()

np.int64(0)

In [16]:
clean.shape

(290584, 5)

Question: does the new page change the percent of visitors who buy? <br>
• <b>Null hypothesis:</b> the page makes no difference. Any difference we see is chance. <br>
• <b>Alternative hypothesis:</b> the new page changes the percent who buy.

Main outcome: converted (bought or not). We have only one outcome, so the normal 0.05 line is used (no stricter line needed).

Was the sample big enough?

1.  Starting value: about 12% of visitors buy with the old page.
2.  Smallest change worth catching: the shop says a rise of 1 point (from 12% to 13%) would pay for the new page.
3.  The 0.05 line.
4.  Power 80%.

In [17]:
needed = NormalIndPower().solve_power(
    effect_size=proportion_effectsize(0.12, 0.13),   # from 12% to 13%
    alpha=0.05,                                      # the 0.05 line
    power=0.8                                        # 80% power
)
print(needed)    # visitors needed in each group

17163.025826592482


Each group has about 145,000 visitors, so the sample is more than big enough.

In [19]:
counts=clean["group"].value_counts()
print(counts)

group
treatment    145310
control      145274
Name: count, dtype: int64


In [22]:
p_share=((counts/counts.sum())*100).round(2)
print(p_share)

group
treatment    50.01
control      49.99
Name: count, dtype: float64


That is a 50/50 split, exactly what a fair coin split should give. The groups are fine.

<b>The professional way:</b> a p-value for the split. The eye cannot always tell a normal split from a broken one (with very big groups, even 49.6% against 50.4% can be a warning sign). So professionals add one line:

In [23]:
print(stats.chisquare(counts).pvalue)    # probability that a size difference this big is just chance

0.9467543681597944


It prints 0.95: the 36-visitor difference is very likely just chance, so the split is fine. <br><br>
How to read it:
p-value of the split check
| p-value | Meaning | What to do |
| :--- | :--- | :--- |
| **0.05 or above** | The size difference is normal chance | Good, continue |
| **Below 0.05** | The size difference is too big to be chance: the split is probably broken | Stop, find the cause before trusting any result |

## <b> The main test </b>

In [24]:
table=pd.crosstab(clean["group"],clean["converted"])
print(table)

converted       0      1
group                   
control    127785  17489
treatment  128046  17264


In [32]:
percent_bought=table[1]/table.sum(axis=1) * 100      # percent who bought in each group
print(percent_bought.round(2))  

group
control      12.04
treatment    11.88
dtype: float64


In [34]:
print(stats.chi2_contingency(table).pvalue)

0.19182228096235662


With the old page, 12.04% of visitors bought. With the new page, 11.88%. The p-value is 0.19, above 0.05.
What this result means. The p-value (0.19) is above 0.05, so the null hypothesis stays: the new page makes no real difference to buying.

 ## <b>Did the result change over the weeks? (novelty effect)

In [37]:
clean["week"] = pd.to_datetime(clean["timestamp"]).dt.isocalendar().week
clean.head()

,user_id,timestamp,group,landing_page,converted,week
0,851104,2017-01-21 22:11:48.556739,control,old_page,0,3
1,804228,2017-01-12 08:01:45.159739,control,old_page,0,2
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0,2
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0,1
4,864975,2017-01-21 01:52:26.210827,control,old_page,1,3


In [41]:
weekly = pd.crosstab(clean["week"],clean["group"],values=clean["converted"], aggfunc="mean") * 100


In [42]:
print(weekly.round(1))

group  control  treatment
week                     
1         11.9       11.8
2         12.0       11.8
3         12.1       12.0
4         12.3       12.1


# <b>Decision

Report sentence: "With the old page 12.04% of visitors bought, with the new page 11.88% (chi-square test p = 0.19). There is no evidence that the new page makes more visitors buy. The groups were balanced (145,274 against 145,310), and the result was the same in every week. 3,893 rows where the visitor saw the wrong page were removed before the analysis."
<br><br>
Decision: keep the old page. The new page costs money and does not bring more buyers.
This is an important lesson: "no difference" is also a useful result. It saved the shop from paying for a change that does not help.